# 04 - Request and Response Design

## 1. Four Parts of a Request

**Definition:** Every request can carry data in four places, and each has a clear job.

| Part | Job | Example |
|---|---|---|
| Path parameter | **Which one** resource | /orders/**5** |
| Query parameter | **Which ones** (filter, sort, page) | /orders?**status=paid&page=2** |
| Body | **The data** to create or update | {"item": "Laptop", "qty": 1} |
| Headers | **Metadata** about the request | Authorization: Bearer <JWT>, Content-Type: application/json |

**Rules:**
- Identify one resource → path
- Filter, sort, paginate → query
- Send data (POST, PUT, PATCH) → body
- Auth token, content type, idempotency key → headers
- Never put passwords or tokens in the URL (they end up in logs)

**Real-world example:** Updating the address of order 5:
PATCH /orders/5, header Authorization: Bearer <JWT>, body {"address": "Hyderabad"}

**Interview line:** "Path says which one, query says which ones, body carries the data, and headers carry metadata like the auth token."

In [ ]:
# ==================================================
# 1. BREAKING A REQUEST INTO ITS FOUR PARTS
# ==================================================
from urllib.parse import urlparse, parse_qs

request = {
    "method": "PATCH",
    "url": "https://api.shopkart.com/orders/5?notify=true",
    "headers": {"Authorization": "Bearer eyJhbGciOi...", "Content-Type": "application/json"},
    "body": {"address": "Hyderabad"},
}

parsed = urlparse(request["url"])
order_id = parsed.path.split("/")[-1]
query = {k: v[0] for k, v in parse_qs(parsed.query).items()}

print("=" * 55)
print(f"[METHOD]  {request['method']}")
print(f"[PATH]    order_id = {order_id}        -> which one")
print(f"[QUERY]   {query}     -> options/filters")
print(f"[BODY]    {request['body']}  -> the data")
print(f"[HEADERS] {request['headers']}")
print("=" * 55)

## 2. Response Design

**Definition:** A good response is **predictable**: the right status code, a consistent JSON shape, and only the fields the client needs.

**Rules:**
- Return the created or updated resource (with its id) after POST, PUT, PATCH
- Use consistent naming (pick snake_case or camelCase and stick to it)
- Never return sensitive fields (password hash, internal IDs, secrets)
- Wrap lists with metadata (total, page, size)
- Use a **response model** to control exactly what goes out (FastAPI: response_model)

**Real-world example:** GET /users/5 returns id, name, email. It never returns password_hash, even though the database row has it.

**Interview line:** "I keep responses consistent, return the resource after writes, use response models to hide sensitive fields, and add metadata like total and page for lists."

In [ ]:
# ==================================================
# 2. RESPONSE DESIGN - hide sensitive fields, wrap lists
# ==================================================
db_user = {"id": 5, "name": "Aleti", "email": "aleti@example.com",
           "password_hash": "$2b$12$abc...", "internal_flag": True}

PUBLIC_FIELDS = ["id", "name", "email"]

# BAD: returning the DB row as-is
print("[BAD]  ", db_user)

# GOOD: only public fields (what response_model does in FastAPI)
safe_user = {k: db_user[k] for k in PUBLIC_FIELDS}
print("[GOOD] ", safe_user)

# GOOD: list wrapped with metadata
orders = [{"id": 1}, {"id": 2}, {"id": 3}]
list_response = {"items": orders, "total": 42, "page": 1, "size": 3}
print("[LIST] ", list_response)

## Key Takeaways

- Path = which one, query = which ones, body = data, headers = metadata
- Never put passwords or tokens in the URL
- Return the resource with its id after writes
- Hide sensitive fields using a response model
- Wrap lists with total, page, size